# Renovación de préstamo: flujo refactorizado

Propensión **comercial** a aceptar una renovación. Este notebook organiza el análisis y llama al paquete `loan_renewal`; la lógica está en módulos probados. No determina elegibilidad crediticia.

El original tiene 186 celdas. Se conservan el problema y las familias de modelos, con correcciones metodológicas documentadas en `docs/refactorizacion.md`. Las métricas no son equivalentes a las tablas del original.


## Correspondencia con las unidades 1–6 del curso

| Unidad | Entrada del proyecto | Función |
|---|---|---|
| 01 | `Unidad01/entorno.py` | Entorno y dependencias |
| 02 | `Unidad02/run.py` | Modularización y entrenamiento |
| 03 | `Unidad03/versionar.py --dvc` | Hash y puntero DVC local |
| 04 | `Unidad04/run_pipeline.py --tune --track` | Etapas y tracking |
| 05 | `docker compose up --build -d` | API `/health` y `/predecir` |
| 06 | `Unidad06/pipeline/run_pipeline.py` | Grid, random, comparación y Registry |

Estas entradas reutilizan los módulos de este notebook. La unidad 6 guarda runs en SQLite local y registra un alias `candidato`; no despliega ni aprueba el modelo. Los comandos por unidad y sus diferencias respecto de la referencia están en `docs/unidades_01_06.md`. Para ejecutarlos, activar `.venv` e instalar `.[dev,api,mlops]`. No publicar CSV, modelos ni bases de tracking en Git.


In [ ]:
from pathlib import Path
import sys, json, logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Abre el notebook dentro del repositorio SDG-MLOPS-USE-CASE-LOAN")
sys.path.insert(0, str(ROOT / "src"))
from loan_renewal.data import load_dataset, validate_dataset, split_temporal
from loan_renewal.schema import FEATURE_COLUMNS, TARGET
from loan_renewal.pipeline import train
from loan_renewal.predict import load_artifact, predict_frame
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
DATA = ROOT / "data/raw/Dataset Renovacion_prestamo.csv"
OUTPUT = ROOT / "artifacts"
if not DATA.exists():
    raise FileNotFoundError(f"Coloca el CSV original en {DATA}")


## 1. Ingestión y contrato

CSV separado por `;`, etiqueta `FLAG_VENTA=1`. `CLIENTE` y `MES` sirven para validación y partición, nunca como predictores. El diccionario llama `USO_TRIM_LINEA_Bco` a la variable `USO_TRIM_LINEA_BBVA` del CSV. La presentación menciona dos meses; el CSV contiene nueve.


In [ ]:
df = load_dataset(DATA)
profile = validate_dataset(df)
display(pd.DataFrame({"conteo": df[TARGET].value_counts().sort_index(), "proporcion": df[TARGET].value_counts(normalize=True).sort_index()}))
display(pd.DataFrame({"nulos": df.isna().sum(), "porcentaje": df.isna().mean().mul(100)}).query("nulos > 0"))
print({k: profile[k] for k in ["rows", "columns", "unique_customers", "months"]})


## 2. Partición antes de aprender transformaciones

Entrenamiento: enero–junio de 2015. Validación: julio–agosto. Test reservado: septiembre. La imputación, escalado, vocabulario categórico y KMeans se ajustan exclusivamente en train. El test se usa para reportar el candidato seleccionado, sin reentrenar con validación.


In [ ]:
parts = split_temporal(df)
display(pd.DataFrame([{"particion": name, "filas": len(frame), "positivos": int(frame[TARGET].sum()), "meses": sorted(frame.MES.unique().tolist())} for name, frame in parts.items()]))


## 3. Exploración de entrenamiento

Los gráficos se calculan sobre train. El análisis histórico del original ya exploró el archivo completo: este corte corrige el nuevo procedimiento, pero una campaña futura será necesaria para una evaluación prospectiva nueva.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
parts["train"][TARGET].value_counts().sort_index().plot.bar(ax=axes[0], color=["#64748b", "#f97316"], title="Distribución de renovación en train")
axes[0].set_xlabel("FLAG_VENTA")
axes[0].set_ylabel("Registros")
np.log1p(parts["train"]["LINEA_RENOVADO"].clip(lower=0)).hist(ax=axes[1], bins=35, color="#2563eb")
axes[1].set_title("Monto ofrecido en train, log1p")
axes[1].set_xlabel("log1p(monto)")
fig.tight_layout()
plt.show()


## 4. Entrenamiento y selección reproducible

Se comparan seis candidatos base: baseline Dummy, árbol original, árbol con under/oversampling y dos Random Forest. Cada candidato tiene su pipeline completo; el muestreo ocurre solo al entrenar. El modelo y el umbral se seleccionan por F1 positivo en validación, con AP como desempate. Además se conserva GridSearchCV de Random Forest con cuatro combinaciones y tres folds temporales internos de train (12 ajustes CV y un refit). Cada fold aprende de nuevo el preprocesamiento y el muestreo. Es una búsqueda acotada, no exhaustiva.

`train` genera `model.joblib`, `metrics.json`, `manifest.json` y `comparison.csv`. SMOTE y XGBoost son opcionales mediante la CLI y requieren sus extras.


In [ ]:
# Cada ejecución reproduce el protocolo fijo; no retocar candidatos usando resultados de test.
ENABLE_TUNING = True  # False para ejecutar solo los seis candidatos base
report = train(DATA, OUTPUT, tune=ENABLE_TUNING)
comparison = pd.read_csv(OUTPUT / "comparison.csv")
display(comparison[["candidate", "sampling", "validation_threshold", "validation_precision", "validation_recall", "validation_f1", "validation_roc_auc", "validation_average_precision"]])
print("Seleccionado:", report["winner"], "umbral:", report["threshold"])


## 5. Evaluación final del seleccionado

ROC-AUC y Average Precision usan scores, no etiquetas duras. La precisión positiva, recall y F1 se calculan con el umbral elegido en validación. La accuracy se interpreta junto al desbalance. No se copian resultados del notebook anterior.


In [ ]:
display(pd.DataFrame({"validation": report["validation"], "test": report["test"]}).drop(index=["confusion_matrix"]))
display(pd.DataFrame(report["test"]["confusion_matrix"], index=["real 0", "real 1"], columns=["predicho 0", "predicho 1"]))
print("Paridad tras recarga:", report["reload_prediction_parity"])


## 6. Inferencia desde variables originales

El artefacto contiene transformaciones, modelo y umbral. No construir columnas `_LOG`, one-hot o clusters manualmente. Se rechazan features ausentes, extras y valores infinitos. Las categorías desconocidas se admiten con el vocabulario aprendido en train.


In [ ]:
bundle = load_artifact(OUTPUT / "model.joblib")
# Comprobación técnica de paridad con entradas del holdout; no es una recomendación de contacto.
raw_examples = parts["test"].loc[:, FEATURE_COLUMNS].head(5)
predictions = predict_frame(bundle, raw_examples)
assert len(predictions) == len(raw_examples)
assert np.isfinite(predictions["probabilidad_renovacion"]).all()
print("Contrato raw validado; inferencia finita para", len(predictions), "registros.")
print("Hash de datos:", report["manifest"]["dataset_sha256"])


## 7. Límites y ejecución

Los scores son salidas sin calibración, especialmente tras balanceo. El rendimiento no demuestra utilidad comercial: faltan costes y capacidad de campaña. Confirmar que las features estaban disponibles en el instante de la oferta. Las variables demográficas se conservan para trazabilidad académica y requieren revisión antes de cualquier uso operativo.

Desde la raíz:

```bash
python -m unittest discover -s tests -v
python -m loan_renewal train --data "data/raw/Dataset Renovacion_prestamo.csv" --output artifacts
```

Ver `docs/refactorizacion.md` para el mapa original→módulos y `docs/diccionario.md` para las 22 columnas. Los datos, originales y artefactos se mantienen fuera de Git.
